# Jackfruit FORK: does a patched fact make the model infer its cause in a fork?
**Fork:** one cause, two effects (cold weather → frozen pond, cold weather → people in thick coats).
* *Seeing* "the pond was frozen" suggests cold weather, so thick coats become more likely.
* *Doing* ("park staff froze the pond with machines") says nothing about the weather, so coats should not change.

We patch the key-word state ("**frozen**") into a story whose pond is "**round**" and check which pattern the model follows.

**Scores (forks).** Under *doing*, neither the cause nor the sibling effect should move; only the patched fact itself does. So the effect size is matched on the **mediator** (the patched fact): r = Δcause / Δmediator. D = (r_see − r_patch) / (r_see − r_do), 0 = seeing, 1 = doing; a patch is scored only if its Δmediator ≥ ½ Δmediator(see). The sibling effect gets the same score (D_sib) as a second read-out.
**Filters (fixed before running):** Δcause(see) > 0.5, Δmediator(see) > 0.5, Δmediator(do) > 0.5, r_see − r_do > 0.25.
**Sources:** true, false, unrelated (another scenario), and **do** (key word from the intervention story). Same pass rule as the chain runs.

15 fork scenarios × 8 held-out variants, Qwen3-4B-Base, word and sentence patches, layers 0–21 every 3rd. Predictions: `prediction_before_FORK.txt`.

In [ ]:
# Kaggle setup: write the story files (embedded below) and work in /kaggle/working/jackfruit
import os, gzip, base64
os.makedirs("/kaggle/working/jackfruit", exist_ok=True)
os.chdir("/kaggle/working/jackfruit")
DATA = {'pairs_fork.csv': '...', 'stories_chain.csv': '...'}
for name, b in DATA.items():
    open(name, "wb").write(gzip.decompress(base64.b64decode(b)))
OUT = "results"
os.makedirs(OUT, exist_ok=True)
import torch
print("working in", os.getcwd(), "| files:", sorted(os.listdir(".")), "| GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ---------- Setup: model, tokenizer, settings
import gc, os, time, torch, numpy as np, pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
torch.set_grad_enabled(False)

NAME = "Qwen/Qwen3-4B-Base"
DEVICE = "cuda"
BS = 16                         # batch size; lower to 8 if you get out-of-memory errors
TRAIN_VARIANTS = range(0, 16)   # used only to learn directions
EVAL_VARIANTS = range(16, 20)   # held out; used for every score. Raise to range(16, 24) for the final run
LAYER_STEP = 4                  # plan of record: every 4th layer
ALPHAS = [1.0, 2.0, 4.0]        # edit strength, in units of the (true - false) difference vector
POSITIONS = ["mediator", "after", "answer"]

for v in ["lm", "model", "W32"]:
    if v in globals():
        del globals()[v]
gc.collect(); torch.cuda.empty_cache()

tok = AutoTokenizer.from_pretrained(NAME)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def load(dtype):
    m = AutoModelForCausalLM.from_pretrained(NAME, torch_dtype=dtype, device_map=DEVICE).eval()
    return m, m.get_output_embeddings().weight.float()     # float32 output layer

lm, W32 = load(torch.float16)
N_LAYERS = lm.config.num_hidden_layers
LAYERS = list(range(0, N_LAYERS, LAYER_STEP))
print("layers to sweep:", LAYERS)

YES = tok(" Yes", add_special_tokens=False).input_ids
NO = tok(" No", add_special_tokens=False).input_ids
assert len(YES) == 1 and len(NO) == 1
YES, NO = YES[0], NO[0]

In [ ]:
# ---------- Prompt building, token positions, belief with optional edit
import ast
FEWSHOT = (
    "Story: The kettle was on the stove and the water started to bubble.\n"
    "Question: Was the water hot?\nAnswer: Yes\n\n"
    "Story: Maya left her umbrella at home and the sky stayed clear all day.\n"
    "Question: Did Maya get wet from rain?\nAnswer: No\n\n"
)
STORY_OFFSET = len(FEWSHOT) + len("Story: ")

def prompt(story, q):
    return f"{FEWSHOT}Story: {story}\nQuestion: {q}\nAnswer:"

def encode(stories, qs, spans):
    """Tokenise a batch (left-padded) and find, per row, the last token of the key sentence."""
    enc = tok([prompt(s, q) for s, q in zip(stories, qs)], return_tensors="pt", padding=True,
              return_offsets_mapping=True)
    offs = enc.pop("offset_mapping")
    am = enc["attention_mask"]
    key_idx = []
    for i, (a, b) in enumerate(spans):
        a, b = a + STORY_OFFSET, b + STORY_OFFSET
        hit = ((offs[i, :, 0] < b) & (offs[i, :, 1] > a) & (am[i] == 1)).nonzero().flatten()
        assert len(hit) > 0, "key sentence not found in tokens"
        key_idx.append(hit[-1].item())
    return enc, torch.tensor(key_idx)

def position_mask(key_idx, T, mode):
    B = len(key_idx)
    m = torch.zeros(B, T)
    for i, k in enumerate(key_idx.tolist()):
        if mode == "mediator":
            m[i, k] = 1
        elif mode == "after":
            m[i, k + 1:T - 1] = 1     # tokens after the key sentence, excluding the final token
        elif mode == "answer":
            m[i, T - 1] = 1
    return m

def _layer_out(out):
    return out[0] if isinstance(out, tuple) else out

def belief(stories, qs, spans, edit=None, bs=BS):
    """log P(Yes) - log P(No). edit = dict(layer=int, vecs=[N, H] tensor, mode=str) or None."""
    res = []
    for i in range(0, len(stories), bs):
        enc, key_idx = encode(stories[i:i+bs], qs[i:i+bs], spans[i:i+bs])
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)
        handle = None
        if edit is not None:
            T = enc["input_ids"].shape[1]
            mask = position_mask(key_idx, T, edit["mode"]).to(DEVICE, lm.dtype)
            vecs = edit["vecs"][i:i+bs].to(DEVICE, lm.dtype)
            def hook(mod, inp, out, mask=mask, vecs=vecs):
                h = _layer_out(out) + mask[:, :, None] * vecs[:, None, :]
                return (h,) + tuple(out[1:]) if isinstance(out, tuple) else h
            handle = lm.model.layers[edit["layer"]].register_forward_hook(hook)
        try:
            h = lm.model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
                         position_ids=pos).last_hidden_state[:, -1].float()
        finally:
            if handle is not None:
                handle.remove()
        logp = torch.log_softmax(h @ W32.T, dim=-1)
        res += (logp[:, YES] - logp[:, NO]).tolist()
    return np.array(res)

def key_states(stories, qs, spans, layers, bs=BS):
    """Residual stream (output of each layer in `layers`) at the last token of the key sentence."""
    out = []
    for i in range(0, len(stories), bs):
        enc, key_idx = encode(stories[i:i+bs], qs[i:i+bs], spans[i:i+bs])
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)
        hs = lm.model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"], position_ids=pos,
                      output_hidden_states=True).hidden_states
        rows = torch.arange(len(key_idx))
        out.append(torch.stack([hs[L + 1][rows, key_idx.to(DEVICE)].float().cpu() for L in layers], 1))
    return torch.cat(out)        # [N, len(layers), H]

# float16 overflow check; fall back to bfloat16 if needed
t = belief(["Priya ate a huge lunch and felt full."], ["Was Priya full?"], [[0, 5]])
if not np.isfinite(t).all():
    print("float16 overflow -> bfloat16")
    del lm, W32; gc.collect(); torch.cuda.empty_cache()
    lm, W32 = load(torch.bfloat16)
print("dtype:", lm.dtype)

d = pd.read_csv("stories_chain.csv")
d["variant"] = d.item_id.str.rsplit("_", n=1).str[1].astype(int)
for c in [c for c in d.columns if c.startswith("span_")]:
    d[c] = d[c].apply(ast.literal_eval)
train = d[d.variant.isin(TRAIN_VARIANTS)].reset_index(drop=True)
ev = d[d.variant.isin(EVAL_VARIANTS)].reset_index(drop=True)
SCEN = sorted(d.scenario_id.unique())
print(f"{len(SCEN)} scenarios | train items {len(train)} | eval items {len(ev)}")

In [ ]:
# ---------- Settings (fork scenarios)
import glob, matplotlib.pyplot as plt
from scipy.stats import wilcoxon
TAG = "FK"
LAYERS = [L for L in [0, 3, 6, 9, 12, 15, 18, 21] if L < N_LAYERS]
MODES = ["word", "sentence"]
KINDS = ["true", "do", "false", "unrelated"]
FILLERS = ["Was there a cat nearby?", "Did someone in the story have a brother?", "Was a radio playing?"]
PROBES = {"cause": "q_cause", "effect": "q_eff", "med": "q_med", **{f"f{i}": f"q_f{i}" for i in range(len(FILLERS))}}

P = pd.read_csv("pairs_fork.csv")
for c in [c for c in P.columns if c.startswith(("span_", "wspan_"))]:
    P[c] = P[c].apply(ast.literal_eval)
P = P[P.variant.between(16, 23)].reset_index(drop=True)
for i, f in enumerate(FILLERS):
    P[f"q_f{i}"] = f

# the key word inside the intervention sentence: last occurrence of the same word in the do story
def do_word_span(r):
    wa, wb = r.wspan_see
    w = r.see[wa:wb]
    k = r.do.rfind(w)
    assert k >= r.span_do[0] and r.do[k + len(w):].startswith("."), (r.item_id, w)
    return [k, k + len(w)]
P["wspan_do"] = P.apply(do_word_span, axis=1)
P["span_do_s"] = P.apply(lambda r: [r.wspan_do[0], r.span_do[1]], axis=1)   # used only for "sentence end" lookup
print(f"{P.scenario_id.nunique()} scenarios, {len(P)} items | layers {LAYERS}")

## Step 1: reference (the model's own seeing and doing from text)

In [ ]:
path = f"{OUT}/{TAG}_reference_items.csv"
if os.path.exists(path):
    ref = pd.read_csv(path)
else:
    ref = P[["scenario_id", "item_id"]].copy()
    for cond in ["base", "see", "do", "false"]:
        for p, q in PROBES.items():
            ref[f"{p}_{cond}"] = belief(P[cond].tolist(), P[q].tolist(), P[f"span_{cond}"].tolist())
    ref.to_csv(path, index=False)
nf = len(FILLERS)
for cond in ["see", "do", "false"]:
    fill = sum(ref[f"f{i}_{cond}"] - ref[f"f{i}_base"] for i in range(nf)) / nf
    ref[f"dfill_{cond}"] = fill
    for p in ["cause", "effect", "med"]:
        ref[f"d{p}C_{cond}"] = ref[f"{p}_{cond}"] - ref[f"{p}_base"] - fill
sc = ref.groupby("scenario_id")[[c for c in ref.columns if c.startswith("d")]].mean()
sc["r_see"] = sc.dcauseC_see / sc.dmedC_see
sc["r_do"] = sc.dcauseC_do / sc.dmedC_do
sc["s_see"] = sc.deffectC_see / sc.dmedC_see          # sibling effect per unit of mediator
sc["s_do"] = sc.deffectC_do / sc.dmedC_do
sc["valid"] = ((sc.dcauseC_see > 0.5) & (sc.dmedC_see > 0.5) & (sc.dmedC_do > 0.5) & (sc.r_see - sc.r_do > 0.25))
sc.to_csv(f"{OUT}/{TAG}_reference_scenarios.csv")
display(sc[["dmedC_see", "dcauseC_see", "deffectC_see", "dmedC_do", "dcauseC_do", "deffectC_do", "dmedC_false", "dfill_see", "r_see", "r_do", "s_see", "s_do", "valid"]].round(2))
VALID = sc.index[sc.valid].tolist()
print(f"valid fork scenarios: {len(VALID)} / {len(sc)}")
P = P[P.scenario_id.isin(VALID)].reset_index(drop=True)
assert len(VALID) >= 3, "too few valid fork scenarios to continue"

## Step 2: record source states (true, do, false, unrelated) and patch

In [ ]:
def key_positions(stories, qs, spans, wspans):
    """Per row: index of the last token of the key word, and of the sentence end."""
    enc = tok([prompt(s, q) for s, q in zip(stories, qs)], return_tensors="pt", padding=True, return_offsets_mapping=True)
    offs = enc.pop("offset_mapping"); am = enc["attention_mask"]
    wl, se = [], []
    for i, ((a, b), (wa, wb)) in enumerate(zip(spans, wspans)):
        def last_hit(x, y):
            x, y = x + STORY_OFFSET, y + STORY_OFFSET
            h = ((offs[i, :, 0] < y) & (offs[i, :, 1] > x) & (am[i] == 1)).nonzero().flatten()
            assert len(h) > 0
            return h[-1].item()
        wl.append(last_hit(wa, wb)); se.append(last_hit(a, b))
    return enc, wl, se

def record(stories, qs, spans, wspans, bs=BS):
    """States at [key-word-last ... sentence-end] for every layer in LAYERS. Returns list of [L, n_pos, H] tensors."""
    out = []
    for i in range(0, len(stories), bs):
        enc, wl, se = key_positions(stories[i:i+bs], qs[i:i+bs], spans[i:i+bs], wspans[i:i+bs])
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)
        hs = lm.model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"], position_ids=pos,
                      output_hidden_states=True).hidden_states
        for r in range(len(wl)):
            out.append(torch.stack([hs[L + 1][r, wl[r]:se[r] + 1].float().cpu() for L in LAYERS]))
    return out

def belief_patch(stories, qs, spans, wspans, src, li, mode, bs=BS):
    """Run the target stories with states replaced at layer LAYERS[li]. src[row] = [L, n_pos, H]."""
    res, skipped = [], 0
    L = LAYERS[li]
    for i in range(0, len(stories), bs):
        enc, wl, se = key_positions(stories[i:i+bs], qs[i:i+bs], spans[i:i+bs], wspans[i:i+bs])
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)
        items = []
        for r in range(len(wl)):
            s_states = src[i + r][li]                         # [n_pos_src, H]
            n_tgt = se[r] - wl[r] + 1
            if mode == "word":
                items.append(([wl[r]], s_states[:1]))
            elif s_states.shape[0] == n_tgt:
                items.append((list(range(wl[r], se[r] + 1)), s_states))
            else:
                items.append(None); skipped += 1
        def hook(mod, inp, out, items=items):
            h = _layer_out(out).clone()
            for r, it in enumerate(items):
                if it is not None:
                    h[r, it[0]] = it[1].to(h.device, h.dtype)
            return (h,) + tuple(out[1:]) if isinstance(out, tuple) else h
        handle = lm.model.layers[L].register_forward_hook(hook)
        try:
            h = lm.model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"], position_ids=pos).last_hidden_state[:, -1].float()
        finally:
            handle.remove()
        logp = torch.log_softmax(h @ W32.T, dim=-1)
        b = (logp[:, YES] - logp[:, NO]).tolist()
        res += [np.nan if it is None else v for it, v in zip(items, b)]
    return np.array(res), skipped


t0 = time.time()
q0 = P.q_med.tolist()
SRC = {"true": record(P.see.tolist(), q0, P.span_see.tolist(), P.wspan_see.tolist()),
       "do": record(P.do.tolist(), q0, P.span_do_s.tolist(), P.wspan_do.tolist()),
       "false": record(P.false.tolist(), q0, P.span_false.tolist(), P.wspan_false.tolist())}
other = {s: VALID[(VALID.index(s) + 1) % len(VALID)] for s in VALID}
lookup = {(r.scenario_id, r.variant): i for i, r in P.iterrows()}
unrel = []
for _, r in P.iterrows():
    j = lookup.get((other[r.scenario_id], r.variant))
    if j is None:                                   # exploration has 4 variants, confirmation 8: fall back to variant 16
        j = lookup[(other[r.scenario_id], 16)]
    unrel.append(j)
U = P.loc[unrel]
SRC["unrelated"] = record(U.see.tolist(), U.q_med.tolist(), U.span_see.tolist(), U.wspan_see.tolist())
print(f"recorded sources for {len(P)} items ({(time.time() - t0) / 60:.1f} min)")

# how different is the do-source state from the true-source state at the key word?
sim = []
for i, s in enumerate(P.scenario_id):
    a, b = SRC["true"][i][:, 0], SRC["do"][i][:, 0]
    cos = torch.nn.functional.cosine_similarity(a, b, dim=-1)
    rel = (a - b).norm(dim=-1) / a.norm(dim=-1)
    for li, L in enumerate(LAYERS):
        sim.append(dict(scenario_id=s, layer=L, cos_true_do=cos[li].item(), reldist_true_do=rel[li].item()))
sim = pd.DataFrame(sim); sim.to_csv(f"{OUT}/{TAG}_state_similarity.csv", index=False)
display(sim.groupby("layer")[["cos_true_do", "reldist_true_do"]].mean().round(3))

stories, spans, wspans = P.base.tolist(), P.span_base.tolist(), P.wspan_base.tolist()
base_b = {p: belief(stories, P[q].tolist(), spans) for p, q in PROBES.items()}
for li, L in enumerate(LAYERS):
    path = f"{OUT}/{TAG}_sweep_layer{L:02d}.csv"
    if os.path.exists(path):
        print("skip layer", L); continue
    rows = []
    for mode in MODES:
        for kind in KINDS:
            r = P[["set", "scenario_id", "item_id"]].copy()
            r["layer"], r["mode"], r["kind"] = L, mode, kind
            for p, q in PROBES.items():
                b, skipped = belief_patch(stories, P[q].tolist(), spans, wspans, SRC[kind], li, mode)
                r[f"d{p}"] = b - base_b[p]
            r["skipped"] = skipped
            rows.append(r)
    pd.concat(rows).to_csv(path, index=False)
    print(f"layer {L} done ({(time.time() - t0) / 60:.1f} min)")

## Step 3: results
1. Pass rule for the true patch and for the do-source patch (each against the unrelated patch).
2. D for true and do-source patches by layer, and the paired difference D(do) − D(true) with bootstrap CIs over scenarios.

In [ ]:
rng = np.random.default_rng(0)
def boot_median(x, n=2000):
    x = pd.Series(x).dropna().values
    if len(x) < 3:
        return np.nan, np.nan
    m = [np.median(rng.choice(x, len(x))) for _ in range(n)]
    return np.percentile(m, 2.5), np.percentile(m, 97.5)

sw = pd.concat(pd.read_csv(f) for f in sorted(glob.glob(f"{OUT}/{TAG}_sweep_layer*.csv")))
sw["dfill"] = sw[[f"df{i}" for i in range(len(FILLERS))]].mean(1)
for p in ["cause", "effect", "med"]:
    sw[f"d{p}C"] = sw[f"d{p}"] - sw.dfill
g = sw.groupby(["scenario_id", "layer", "mode", "kind"])[["dcauseC", "deffectC", "dmedC", "dfill"]].mean().reset_index()
g.to_csv(f"{OUT}/{TAG}_scenario_means.csv", index=False)
print("items skipped in sentence mode:", sw[sw["mode"] == "sentence"].groupby("kind").skipped.max().to_dict())

rows = []
for (L, mode), grp in g.groupby(["layer", "mode"]):
    u = grp[grp.kind == "unrelated"].set_index("scenario_id"); f = grp[grp.kind == "false"].set_index("scenario_id")
    for kind in ["true", "do"]:
        t = grp[grp.kind == kind].set_index("scenario_id")
        diff = (t.dmedC - u.dmedC).dropna()
        rows.append(dict(layer=L, mode=mode, kind=kind, dmedC=t.dmedC.mean(), dcauseC=t.dcauseC.mean(), deffectC=t.deffectC.mean(),
                         dfill=t.dfill.mean(), dmedC_unrel=u.dmedC.mean(), dmedC_false=f.dmedC.mean(),
                         minus_unrel=diff.mean(), share_wins=(diff > 0).mean(),
                         passes=bool((t.dmedC.mean() >= 1.0) and (diff.mean() > 0.5) and ((diff > 0).mean() >= 0.7)
                                     and (abs(t.dfill.mean()) < 0.5) and (f.dmedC.mean() < 0))))
res = pd.DataFrame(rows); res.to_csv(f"{OUT}/{TAG}_settings.csv", index=False)
display(res.round(2))

t = g[g.kind.isin(["true", "do"])].merge(sc[["dmedC_see", "r_see", "r_do", "s_see", "s_do"]], left_on="scenario_id", right_index=True)
t["works"] = t.dmedC >= 0.5 * t.dmedC_see
t["D"] = np.where(t.works, (t.r_see - t.dcauseC / t.dmedC) / (t.r_see - t.r_do), np.nan)
t["D_sib"] = np.where(t.works & ((t.s_see - t.s_do).abs() > 0.1), (t.s_see - t.deffectC / t.dmedC) / (t.s_see - t.s_do), np.nan)
t.to_csv(f"{OUT}/{TAG}_scenario_scores.csv", index=False)
w = t.pivot_table(index=["scenario_id", "layer", "mode"], columns="kind", values="D").reset_index()
out = []
for (L, mode), grp in t.groupby(["layer", "mode"]):
    row = dict(layer=L, mode=mode)
    for kind in ["true", "do"]:
        x = grp[grp.kind == kind]
        lo, hi = boot_median(x.D)
        row.update({f"D_{kind}": x.D.median(), f"D_{kind}_lo": lo, f"D_{kind}_hi": hi, f"n_{kind}": int(x.D.notna().sum()),
                    f"works_{kind}": x.works.mean()})
        row[f"Dsib_{kind}"] = x.D_sib.median()
    pw = w[(w.layer == L) & (w["mode"] == mode)]
    d = (pw["do"] - pw["true"]) if {"do", "true"} <= set(pw.columns) else pd.Series(dtype=float)
    lo, hi = boot_median(d)
    row.update(dict(D_do_minus_true=d.median(), diff_lo=lo, diff_hi=hi, n_paired=int(d.notna().sum())))
    out.append(row)
dres = pd.DataFrame(out).merge(res.pivot_table(index=["layer", "mode"], columns="kind", values="passes").add_prefix("passes_").reset_index(),
                               on=["layer", "mode"])
dres.to_csv(f"{OUT}/{TAG}_D_summary.csv", index=False)
display(dres.round(2))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, mode in zip(axes[:2], MODES):
    x = dres[dres["mode"] == mode]
    for kind, c in [("true", "C0"), ("do", "C3")]:
        ok = x[x[f"n_{kind}"] >= 5]
        ax.errorbar(ok.layer, ok[f"D_{kind}"], yerr=[ok[f"D_{kind}"] - ok[f"D_{kind}_lo"], ok[f"D_{kind}_hi"] - ok[f"D_{kind}"]],
                    fmt="o-", color=c, capsize=3, label=f"{kind}-source patch")
    ax.axhline(0, c="k", lw=.5, ls="--"); ax.axhline(1, c="k", lw=.5, ls="--")
    ax.set_title(f"D by layer, {mode} patch (n >= 5)"); ax.set_xlabel("layer"); ax.set_ylabel("median D (95% CI)"); ax.legend(fontsize=8)
m = g[g["mode"] == "word"].groupby(["kind", "layer"]).dmedC.mean().unstack(0)
for kind, c in [("true", "C0"), ("do", "C3"), ("unrelated", "C7"), ("false", "C2")]:
    axes[2].plot(m.index, m[kind], "o-", color=c, label=kind)
axes[2].axhline(0, c="k", lw=.5); axes[2].set_title("mediator shift, word patch"); axes[2].set_xlabel("layer"); axes[2].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/{TAG}_results.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/results_FORK", "zip", OUT)
print("saved /kaggle/working/results_FORK.zip")